# DM-2 : Boosting et cartes de saillance

Importation des librairies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import joblib
from pathlib import Path

MODELS = Path("models")
MODELS.mkdir(exist_ok=True)
Path("images").mkdir(exist_ok=True)

def fit_or_load(name, model, X, y):
    """Recharge le modèle s'il a déjà été entraîné, sinon l'entraîne et le sauvegarde.
    Attention : le nom sert de clé de cache. Si on change les paramètres, il faut changer le nom
    (ou supprimer le fichier correspondant dans models/)."""
    path = MODELS / f"{name}.joblib"
    if path.exists():
        return joblib.load(path)
    model.fit(X, y)
    joblib.dump(model, path, compress=3)
    return model

## 0. Import du dataset MNIST

### A faire
- Des matrices de confusion manquent. Le sujet en demande une « pour chaque combinaison de paramètres ». Vous n’en tracez pas pour les cas binaires, et pour le multiclasse vous n’en avez qu’une par type de classifieur faible.
- Le nombre d’étapes de boosting n’est pas vraiment étudié. Le meilleur n_estimators vaut toujours 100, la borne haute de votre grille : l’accuracy augmente donc encore à 100. Le sujet demande de faire varier ce paramètre. Une courbe accuracy en fonction de n_estimators pour chaque classifieur faible serait plus parlante que le seul meilleur score. Ces valeurs sont déjà calculées dans grid.cv_results_["mean_test_score"].
- Le cas 0–4 contre 5–9, cité dans le sujet, n’est pas encore traité.

### A corriger
- La première matrice de confusion a pour texte alternatif « décision stump », alors que le modèle utilise des arbres de profondeur 20.
- Dans la grid search, le titre de chaque matrice affiche l’accuracy de validation croisée, alors que la matrice est calculée sur le jeu de test.

In [ ]:
import sklearn
sklearn.__version__ #vérifier qu'on a bien la même version de sklearn pour que nos modèles puissent être réutilisés

In [ ]:
from sklearn.datasets import fetch_openml

X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X = X / 255.0
y = y.astype(int)

Pour `AdaBoost`, il n'est pas vraiment nécessaire de diviser `X` par $255.0$. Cependant, cela devient nécessaire pour le perceptron. 

Par ailleurs, nous n'utilisons pas `StandardScaler` pour éviter d'amplifier le bruit du fond (variance presque nulle) et éviter toute fuite de données de `test` à `train`.

In [ ]:
print(X.shape, y.shape)
plt.imshow(X[0, :].reshape(28, 28), cmap="Greys_r")

In [ ]:
np.unique_counts(y)

- Le jeu de données comprend 70000 images aplaties de taille (1, 784) ((28,28) à l'origine).
- Ce sont des chiffres manuscrits entre 0 et 9, les classes peuvent être considérées étant équilibrées.

## 1. Utilisation d’un algorithme de boosting sur de la reconnaissance de chiﬀres manuscrits

Séparation du dataset en set d'entraînement et de test. \
75 % en entraînement et 25% en test, car nous avons un grand nombre d'images.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### A. Première modélisation

Première modelisation avec Adaboost :
- Abre de décision (profondeur maximal de 20)
- nombre d'estimateur : 20
- learning rate : 0.5

Avec une profondeur macimale de $20$, nous ne pouvons pas vraiment dire que c'est un apprenant faible. Nous avons repris les hyperparamètres du site Medium donné dans le sujet du DM.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

Tree_adaboost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=20),
                                  n_estimators=20,
                                  learning_rate=0.5,
                                  random_state=42)

Tree_adaboost.fit(X_train, y_train)

Tree_adaboost_score = Tree_adaboost.score(X_test, y_test)
print(f"AdaBoost score: {Tree_adaboost_score}")

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

cm_Tree_adaboost = confusion_matrix(y_test, Tree_adaboost.predict(X_test))
cm_Tree_adaboost_disp = ConfusionMatrixDisplay(cm_Tree_adaboost)
cm_Tree_adaboost_disp.plot()
cm_Tree_adaboost_disp.ax_.set_title("20 arbres de profondeur 20. Accuracy = {:.4f}".format(Tree_adaboost_score))
plt.savefig("images/cm_Tree_adaboost.png", dpi=300, bbox_inches="tight")

Avec ce premier modèle, nous obtenons un score d'exactitude (*accuracy*) $0,9644$ en test. La matrice de confusion montre de bons résultats avec une majorité de bons classements, surtout pour les 1.

La matrice de confusion se lit par ligne : chaque ligne correspond à la vraie classe et chaque colonne à la classe prédite. La grande majorité des images se trouve sur la diagonale. Les classes étant de tailles légèrement différentes, on compare les taux de bonne classification (rappel) plutôt que les effectifs bruts :
- les mieux reconnus sont les 1 ($98{,}8\,\%$), les 6 ($98{,}3\,\%$) et les 0 ($98{,}1\,\%$) ;
- les moins bien reconnus sont les 9 ($94{,}2\,\%$), les 8 ($94{,}8\,\%$) et les 3 ($94{,}9\,\%$).

Les principales confusions sont :
- entre les 4 et les 9, dans les deux sens (27 + 25 = 52 erreurs), un 4 fermé en haut ressemblant à un 9 ;
- des 3 prédits comme des 2 (30 erreurs) ;
- des 5 prédits comme des 3 (24 erreurs), ces deux chiffres partagent effectivement la même boucle inférieure.

### B. Gridsearch en variant les classifieurs faibles

Stratification pour la cross-validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = skf.split(X_train, y_train)

Réglage des paramètres à tester en GridSearch.

In [ ]:
from sklearn.linear_model import Perceptron

estimators = {"stump": DecisionTreeClassifier(max_depth=1),
              "tree_d2": DecisionTreeClassifier(max_depth=2),
              "tree_d3": DecisionTreeClassifier(max_depth=3),
              "tree_d5": DecisionTreeClassifier(max_depth=5),
              "tree_d20": DecisionTreeClassifier(max_depth=20),
              "perceptron": Perceptron(max_iter=1000, tol=1e-3, random_state=42)}

params = {"n_estimators" : [5, 10, 20]}

Nous réalisons la cross-validation GridSearch :

In [ ]:
from sklearn.model_selection import GridSearchCV

results = []

for estimator_name, estimator in estimators.items():
    ada_boost = AdaBoostClassifier(estimator=estimator, random_state=42)

    # le nom du cache contient la grille : changer params ne recharge pas un ancien modèle
    grid_name = f"grid_{estimator_name}_" + "_".join(str(n) for n in params["n_estimators"])
    grid = fit_or_load(grid_name,
                       GridSearchCV(estimator=ada_boost, param_grid=params,
                                    scoring="accuracy", cv=skf, n_jobs=-1),
                       X_train, y_train)

    best = grid.best_estimator_
    n_best = grid.best_params_["n_estimators"]
    acc_cv = grid.best_score_
    acc_test = best.score(X_test, y_test)

    # Résultats : accuracy de validation croisée (sert au choix de n_estimators) et accuracy test
    results.append({"estimator": estimator_name, "n_estimators": n_best,
                    "accuracy_cv": acc_cv, "accuracy_test": acc_test})
    print(results[-1])

    # Matrice de confusion sur le jeu de test (le titre affiche donc l'accuracy test)
    disp = ConfusionMatrixDisplay.from_predictions(y_test, best.predict(X_test))
    disp.ax_.set_title(f"{n_best} × {estimator_name}. Accuracy test = {acc_test:.4f}")
    plt.savefig(f"images/cm_{estimator_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
df_results = pd.DataFrame(results, columns=["estimator", "n_estimators", "accuracy_cv", "accuracy_test"])
df_results

<img src="images/cm_stump.png" alt= "Matrice de confusion décision stump" width="500">

<img src="images/cm_tree_d2.png" alt= "Matrice de confusion arbre de profondeur 2" width="500">

<img src="images/cm_tree_d3.png" alt= "Matrice de confusion arbre de profondeur 3" width="500">

<img src="images/cm_tree_d5.png" alt= "Matrice de confusion arbre de profondeur 5" width="500">

<img src="images/cm_tree_d20.png" alt= "Matrice de confusion arbre de profondeur 20" width="500">

<img src="images/cm_perceptron.png" alt= "Matrice de confusion perceptron" width="500">

#### Influence du nombre d'étapes de boosting

Dans la grid search ci-dessus, le meilleur `n_estimators` est toujours la borne haute de la grille : l'accuracy n'a pas encore atteint son plateau. Nous étudions donc le nombre d'étapes de boosting jusqu'à `N_MAX = 100`.

Plutôt que de réentraîner un modèle pour chaque valeur, nous entraînons **un seul** AdaBoost à `N_MAX` étapes par classifieur faible, puis nous utilisons `staged_predict`, qui donne les prédictions après 1, 2, …, `N_MAX` étapes. Avec le même `random_state`, les `k` premières étapes d'un modèle à 100 étapes sont exactement celles d'un modèle entraîné avec `n_estimators = k`. Un seul entraînement donne donc :
- la courbe de l'accuracy test en fonction du nombre d'étapes, pour chaque classifieur faible ;
- une matrice de confusion pour **chaque combinaison** (classifieur faible, `n_estimators`), avec `n_estimators` dans `N_LIST`.

Aucun hyperparamètre n'est choisi sur le jeu de test : ces accuracies servent seulement à comparer les configurations.

La même fonction est réutilisée pour les classes binaires (partie C).

In [ ]:
N_MAX = 100                      # nombre maximal d'étapes de boosting
N_LIST = [5, 10, 20, 50, 100]    # valeurs de n_estimators pour lesquelles on trace une matrice de confusion

def etudier_n_estimators(cas, X_tr, y_tr, X_te, y_te, labels=None):
    """Pour chaque classifieur faible de `estimators` :
    - entraîne (ou recharge) un AdaBoost à N_MAX étapes ;
    - trace une matrice de confusion sur le test pour chaque n de N_LIST ;
    puis trace la courbe accuracy test = f(nombre d'étapes) pour tous les classifieurs faibles.
    Renvoie un DataFrame (classifieur faible × n_estimators) des accuracies sur le test."""
    tableau, courbes = {}, {}
    taille_police = 12 if len(np.unique(y_te)) == 2 else 7

    for estimator_name, estimator in estimators.items():
        ada = fit_or_load(f"ada_{cas}_{estimator_name}_{N_MAX}",
                          AdaBoostClassifier(estimator=estimator, n_estimators=N_MAX, random_state=42),
                          X_tr, y_tr)

        # prédictions après 1, 2, ..., N_MAX étapes (moins si AdaBoost s'est arrêté avant)
        preds = list(ada.staged_predict(X_te))
        n_eff = len(preds)
        courbes[estimator_name] = [np.mean(p == y_te) for p in preds]

        # une matrice de confusion par combinaison (classifieur faible, n_estimators)
        fig, axes = plt.subplots(1, len(N_LIST), figsize=(4.5 * len(N_LIST), 4.5))
        tableau[estimator_name] = {}
        for ax, n in zip(axes, N_LIST):
            k = min(n, n_eff)
            acc = courbes[estimator_name][k - 1]
            tableau[estimator_name][n] = acc
            ConfusionMatrixDisplay.from_predictions(y_te, preds[k - 1], display_labels=labels, ax=ax,
                                                    colorbar=False, text_kw={"fontsize": taille_police})
            titre = f"n_estimators = {n}" if k == n else f"n_estimators = {n} (arrêt à {n_eff})"
            ax.set_title(f"{titre}\naccuracy test = {acc:.4f}")
        fig.suptitle(f"Cas {cas} : AdaBoost + {estimator_name}")
        fig.tight_layout()
        fig.savefig(f"images/cm_{cas}_{estimator_name}.png", dpi=200, bbox_inches="tight")
        plt.show()

    # courbe accuracy test = f(nombre d'étapes de boosting)
    plt.figure(figsize=(8, 5))
    for estimator_name, scores in courbes.items():
        plt.plot(range(1, len(scores) + 1), scores, label=estimator_name)
    plt.xlabel("Nombre d'étapes de boosting (n_estimators)")
    plt.ylabel("Accuracy sur le test")
    plt.title(f"Cas {cas} : accuracy en fonction du nombre d'étapes")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"images/accuracy_vs_n_estimators_{cas}.png", dpi=300, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(tableau).T.rename_axis(index="classifieur faible", columns="n_estimators")

In [ ]:
acc_multi = etudier_n_estimators("multi", X_train, y_train, X_test, y_test)
acc_multi.round(4)

**Analyse** *(à compléter après exécution)* :
- valeur de `n_estimators` à partir de laquelle chaque courbe atteint son plateau ;
- classifieurs faibles qui continuent à progresser à 100 étapes (probablement les plus faibles : stump, arbres de profondeur 2 et 3) ;
- évolution des confusions 4/9, 3/5 et 3/8 quand `n_estimators` augmente.

### C. Etude de classes binaires

Nous étudions ici des problèmes binaires :
- **9 contre 4** et **5 contre 3** : les deux paires les plus confondues par le premier modèle (partie A) ;
- **8 contre 3** : la paire citée dans le sujet, également la plus simple à interpréter pour les cartes de saillance de la partie 2 ;
- **5–9 contre 0–4** : le cas « groupes de chiffres » cité dans le sujet.

Pour que les résultats soient comparables avec le cas multiclasse, nous gardons le **même découpage train/test** que précédemment : nous filtrons `X_train` et `X_test` au lieu de refaire un `train_test_split` (aucune image de test du multiclasse ne se retrouve ainsi dans un train binaire).

Pour chaque cas, nous faisons varier le classifieur faible et le nombre d'étapes de boosting, avec la fonction `etudier_n_estimators` définie en partie B : une matrice de confusion par combinaison, puis la courbe de l'accuracy test. Par convention, la classe 1 est la première citée (9 dans « 9 contre 4 »).

In [ ]:
def sous_probleme(classes_pos, classes_neg):
    """Restreint le découpage train/test aux classes_pos ∪ classes_neg.
    Étiquette 1 pour les classes_pos, 0 pour les classes_neg."""
    classes = classes_pos + classes_neg
    garde_tr, garde_te = np.isin(y_train, classes), np.isin(y_test, classes)
    return (X_train[garde_tr], np.isin(y_train[garde_tr], classes_pos).astype(int),
            X_test[garde_te], np.isin(y_test[garde_te], classes_pos).astype(int))

#### 1. 9 vs 4

In [ ]:
X_9v4_train, y_9v4_train, X_9v4_test, y_9v4_test = sous_probleme([9], [4])
print("train :", X_9v4_train.shape, np.bincount(y_9v4_train), "| test :", X_9v4_test.shape, np.bincount(y_9v4_test))

acc_9v4 = etudier_n_estimators("9v4", X_9v4_train, y_9v4_train, X_9v4_test, y_9v4_test, labels=["4", "9"])
acc_9v4.round(4)

**Analyse** *(à compléter après exécution)* : le 4 fermé en haut ressemble à un 9 : comparer avec les 52 confusions 4/9 de la partie A.

#### 2. 5 vs 3

In [ ]:
X_5v3_train, y_5v3_train, X_5v3_test, y_5v3_test = sous_probleme([5], [3])
print("train :", X_5v3_train.shape, np.bincount(y_5v3_train), "| test :", X_5v3_test.shape, np.bincount(y_5v3_test))

acc_5v3 = etudier_n_estimators("5v3", X_5v3_train, y_5v3_train, X_5v3_test, y_5v3_test, labels=["3", "5"])
acc_5v3.round(4)

**Analyse** *(à compléter après exécution)* : les deux chiffres partagent la boucle inférieure ; la différence se joue dans la partie haute.

#### 3. 8 vs 3

In [ ]:
X_8v3_train, y_8v3_train, X_8v3_test, y_8v3_test = sous_probleme([8], [3])
print("train :", X_8v3_train.shape, np.bincount(y_8v3_train), "| test :", X_8v3_test.shape, np.bincount(y_8v3_test))

acc_8v3 = etudier_n_estimators("8v3", X_8v3_train, y_8v3_train, X_8v3_test, y_8v3_test, labels=["3", "8"])
acc_8v3.round(4)

**Analyse** *(à compléter après exécution)* : le 8 ferme les deux boucles à gauche, contrairement au 3 : peu de pixels devraient suffire.

#### 4. 5–9 vs 0–4

In [ ]:
X_59v04_train, y_59v04_train, X_59v04_test, y_59v04_test = sous_probleme([5, 6, 7, 8, 9], [0, 1, 2, 3, 4])
print("train :", X_59v04_train.shape, np.bincount(y_59v04_train), "| test :", X_59v04_test.shape, np.bincount(y_59v04_test))

acc_59v04 = etudier_n_estimators("59v04", X_59v04_train, y_59v04_train, X_59v04_test, y_59v04_test, labels=["0–4", "5–9"])
acc_59v04.round(4)

**Analyse** *(à compléter après exécution)* : chaque classe regroupe des formes très différentes ; comparer le nombre d'étapes nécessaire avec celui des paires de chiffres.

#### Récapitulatif

In [ ]:
recap = pd.DataFrame({"0–9 (multiclasse)": acc_multi[N_MAX],
                      "9 vs 4": acc_9v4[N_MAX],
                      "5 vs 3": acc_5v3[N_MAX],
                      "8 vs 3": acc_8v3[N_MAX],
                      "5–9 vs 0–4": acc_59v04[N_MAX]})
print(f"Accuracy sur le test après {N_MAX} étapes de boosting")
recap.round(4)

### D. Bilan

**Ce que montrent déjà les résultats obtenus.**
- *Multiclasse, 20 étapes (validation croisée).* L'accuracy croît avec la complexité du classifieur faible : stump (0,49), arbres de profondeur 2 (0,70), 3 (0,78), 5 (0,87) et 20 (0,96). Le perceptron (0,84) se place entre les arbres de profondeur 3 et 5.
- *Le stump est très défavorable en multiclasse.* Un stump ne coupe l'espace qu'en deux, sur un seul pixel : à chaque étape, il ne peut séparer qu'un groupe de chiffres d'un autre. Il faut donc beaucoup d'étapes pour distinguer 10 classes.
- *Les arbres de profondeur 20 donnent le meilleur score*, mais, comme indiqué en partie A, ce ne sont plus vraiment des classifieurs faibles.

**À compléter après exécution des parties B et C :**
- nombre d'étapes nécessaire pour atteindre le plateau, selon le classifieur faible ;
- écart entre les problèmes binaires et le multiclasse (on s'attend à ce que le stump soit beaucoup plus compétitif en binaire) ;
- paire la plus difficile parmi 9/4, 5/3 et 8/3, à mettre en relation avec les confusions de la partie A ;
- difficulté du cas 5–9 contre 0–4 par rapport aux paires de chiffres : chaque groupe mélange des formes très différentes.

**Configurations les plus favorables :** *(à compléter)*

**Configurations les plus défavorables :** *(à compléter)*

## 2. Réalisation de cartes de saillance